<a href="https://colab.research.google.com/github/09Rumaisa/MediBot/blob/main/MediBot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install dotenv langchain pydantic langchain_openai
!pip install -U langchain-community
!pip install chromadb

# Note : libraries will be installed depending on your requirement

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.4/65.4 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 5.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.3/19.3 MB 133.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 107.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 129.3 MB/s eta 0:0

In [2]:
# Imports
from langchain.document_loaders import WebBaseLoader
from langchain.document_loaders import CSVLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain_openai import ChatOpenAI
from langchain.memory import ConversationBufferMemory
import os

##  Step 2: Set Up Keys
 You’ll need:
- An OpenAI API key (for LLM calls)/ Groq API
- A LangSmith API key (for logging and observability)

In [4]:
from google.colab import userdata

smith_key = userdata.get('langsmithkey')
aikey = userdata.get('openai')

import os

if not smith_key:
    os.environ["LANGCHAIN_TRACING_V2"] = "false"
    print("⚠ LangSmith key not found. Tracing disabled.")
else:
  os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
  os.environ["LANGCHAIN_API_KEY"] = smith_key.strip()  # strip to remove hidden newline/spaces
  os.environ["LANGCHAIN_TRACING_V2"] = "true"
  project_name = "MediBot"
  os.environ["LANGCHAIN_PROJECT"] = project_name
  os.environ["OPENAI_API_KEY"] = aikey.strip()  # Also good practice to strip this
  print("✅ LangSmith tracing enabled under project:", project_name)

memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)


✅ LangSmith tracing enabled under project: MediBot


<ipython-input-4-1974209618>:20: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)


##  Step 3: Define Custom Tools
 Build at least one tool (e.g., API, calculator, mock database lookup). Use decorators or `Tool` wrappers.

In [6]:
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
from langchain.agents import tool
# Setup once (outside the tool)
loader = CSVLoader(file_path="dataset.csv")
data = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
splits = text_splitter.split_documents(data)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(splits, embeddings, persist_directory="symptom_index")

memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)


@tool
def check_symptoms_conversational(symptoms: str) -> str:
    """Check symptoms and continue conversation to clarify before concluding."""
    qa_chain = ConversationalRetrievalChain.from_llm(
        llm=ChatOpenAI(model="gpt-4.1-nano"),
        retriever=vectorstore.as_retriever(),
        memory=memory
    )
    return qa_chain.run({"question": symptoms})


<ipython-input-6-348646554>:9: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datas

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
#check_symptoms(" have nausea. cold shivverings are also felt. is there any diseaese I might have")

In [36]:
from langchain_community.tools.tavily_search import TavilySearchResults
tavily_key = userdata.get('tavil')
if not tavily_key:
    print("⚠ Tavily API key not found. Please add it to Colab secrets as 'TAVILY_API_KEY'.")
else:
    os.environ["TAVILY_API_KEY"] = tavily_key.strip()

tavily_tool = TavilySearchResults(k=3)  # You can change `k` for more/less results



@tool
def get_drug_info(drug: str) -> str:
    """Gives information related to a drug from drugs.com. Falls back to Tavily search if not found."""
    url = f"https://www.drugs.com/{drug.lower()}.html"

    try:
        loader = WebBaseLoader(url)
        data = loader.load()

        text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
        all_splits = text_splitter.split_documents(data)

        embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
        vectorstore = Chroma.from_documents(documents=all_splits, embedding=embeddings, persist_directory="./drugdb")

        llm = ChatOpenAI(model="gpt-4.1-nano")
        retriever = vectorstore.as_retriever()

        qa_chain = RetrievalQA.from_chain_type(
            llm=llm,
            chain_type="stuff",
            retriever=retriever,
            return_source_documents=True,
            chain_type_kwargs={
                "prompt": ChatPromptTemplate.from_template("""
You are a helpful medical assistant. Use the information below to answer the question.

If the information is not sufficient, simply say: "No information found."

Context: {context}

Question: {question}

Helpful Answer:""")
            }
        )

        result = qa_chain.invoke(drug)["result"]

        # Check if answer was insufficient
        if "No information found" in result or result.strip() == "":
            raise ValueError("Insufficient info")

        return result

    except Exception as e:
        # Fallback to Tavily search
        try:
            print("⚠ Falling back to Tavily search...")
            query = f"{drug} drug information site:drugs.com OR site:mayoclinic.org OR site:rxlist.com"
            results = tavily_tool.run(query)

            if results:
                return "\n\n".join([r["content"] for r in results])
            else:
                return f"No relevant info found for '{drug}' even via fallback."
        except Exception as fallback_error:
            return f"⚠ Failed to fetch info for '{drug}'. Reason: {str(e)} | Fallback error: {str(fallback_error)}"


In [38]:
get_drug_info("what is arinac for")

⚠ Falling back to Tavily search...


'This product is available in the following dosage forms:\n\n## Before Using\n\nIn deciding to use a medicine, the risks of taking the medicine must be weighed against the good it will do. This is a decision you and your doctor will make. For this medicine, the following should be considered:\n\n### Allergies [...] Using this medicine with any of the following may cause an increased risk of certain side effects but may be unavoidable in some cases. If used together, your doctor may change the dose or how often you use this medicine, or give you special instructions about the use of food, alcohol, or tobacco.\n\n### Other Medical Problems\n\nThe presence of other medical problems may affect the use of this medicine. Make sure you tell your doctor if you have any other medical problems, especially: [...] ![](https://assets.mayoclinic.org/content/dam/media/global/images/2025/03/19/1710753_3980687_0060.png)\n![](https://assets.mayoclinic.org/content/dam/media/global/images/2023/06/26/cont-

In [9]:
%pip install -qU langchain-tavily

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.5/12.5 MB 143.6 MB/s eta 0:00:00


In [10]:
from langchain.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from google.colab import userdata
import os

# Get Tavily API key from Colab secrets and set as environment variable
tavily_key = userdata.get('tavil')
if not tavily_key:
    print("⚠ Tavily API key not found. Please add it to Colab secrets as 'TAVILY_API_KEY'.")
else:
    os.environ["TAVILY_API_KEY"] = tavily_key.strip()

# Instantiate Tavily search tool
tavily_tool = TavilySearchResults(k=3)

@tool
def get_first_aid_info(emergency: str) -> str:
    """
    Fetches offline-friendly summaries and protocols for first aid emergencies like burns, choking, or seizures.
    """
    query = f"first aid protocol for {emergency} site:mayoclinic.org OR site:nhs.uk OR site:redcross.org"
    results = tavily_tool.run(query)
    return "\n\n".join([r["content"] for r in results])

<ipython-input-10-763349398>:14: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-tavily package and should be used instead. To use it run `pip install -U :class:`~langchain-tavily` and import as `from :class:`~langchain_tavily import TavilySearch``.
  tavily_tool = TavilySearchResults(k=3)


In [ ]:
get_first_aid_info("choking")

<ipython-input-11-3652383073>:1: LangChainDeprecationWarning: The method `BaseTool.__call__` was deprecated in langchain-core 0.1.47 and will be removed in 1.0. Use :meth:`~invoke` instead.
  get_first_aid_info("choking")


"Then give yourself abdominal thrusts, also called the Heimlich maneuver, to remove the stuck object. First aid for a choking person First aid for a choking person If a person is choking and cannot talk, cry or laugh forcefully, the American Red Cross recommends this approach to delivering first aid. If a person is choking and cannot talk, cry or laugh forcefully, the American Red Cross recommends this approach to delivering first aid. If an object blocks the airway and causes choking, give first aid. Then give yourself abdominal thrusts, also called the Heimlich maneuver, to remove the stuck object. If you are a Mayo Clinic patient, we will only use your protected health information as outlined in our Notice of Privacy Practices.\n\n![First aid for a choking person](/-/media/kcms/gbs/patient-consumer/images/2017/09/22/17/06/composite-five-and-five-heimlich-8col.jpg)\n\n### First aid for a choking person\n\nIf a person is choking and cannot talk, cry or laugh forcefully, give five back

In [11]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI  # You can replace with DeepSeek, Groq, etc.

# Define a general triage prompt
triage_prompt = ChatPromptTemplate.from_template(
    """You are a helpful and careful virtual medical assistant. A user will describe their symptoms in free text.

Your task:
1. Gently assess the symptoms.
2. Provide general medical advice based on commonly known guidelines (do not diagnose).
3. Warn the user if the symptoms might indicate a serious condition.
4. Encourage the user to visit a doctor if anything sounds unusual, worsening, or concerning.

Patient input:
"{input}"

Now respond with a short and clear message in under 4 sentences."""
)

# Connect the chain
llm = ChatOpenAI(model="gpt-4.1-nano")  # Replace with any other model if needed
triage_chain = triage_prompt | llm | StrOutputParser()

# Example input
user_input = "I've been coughing for three days and have a runny nose."

# Invoke the agent
response = triage_chain.invoke({"input": user_input})
print(response)
@tool
def general(input: str) -> str:
    """Give general medical advice based on user-described symptoms (no diagnosis)."""
    return triage_chain.invoke({"input": input})

It sounds like you may have a common cold or other mild respiratory infection, which often improves with rest and fluids. However, if your cough worsens, you develop difficulty breathing, a high fever, or if your symptoms last longer than a week, it's important to see a healthcare professional. If you experience any chest pain, severe weakness, or other concerning signs, seek medical attention promptly. Take care and monitor your symptoms carefully.


In [ ]:
general("my chest hurts and I feel dizzy ")

"I'm sorry you're experiencing chest pain and dizziness. These symptoms can sometimes be serious and need prompt attention. Please seek immediate medical care or call emergency services to ensure your safety. If your symptoms worsen or you experience additional issues like shortness of breath, nausea, or sweating, go to the emergency room right away."

##  Step 4: Enable Function-Calling
 Define structured functions for your agent to call. Use `openai.FunctionsAgent` or LangChain's `OPENAI_FUNCTIONS` agent type.

In [17]:
from langchain.agents import initialize_agent
from langchain.agents import AgentType
from langchain.chat_models import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-nano")  # Replace with any other model if needed
agent = initialize_agent(
    tools=[general, check_symptoms_conversational,get_drug_info,get_first_aid_info],
    llm=llm,
    agent=AgentType.OPENAI_FUNCTIONS,
    #memory=memory,
    verbose=True,
)


# Try asking:
# agent.run("I have a sore throat and mild fever.")
# agent.run("My eyes are red and I also have a runny nose, what does the dataset say?")


##  Step 6: Fallback Strategy
 Define a fallback agent or tool to be used when the main tool or agent fails.

In [14]:
from langchain.agents.agent import AgentExecutor
from langchain.agents import tool

@tool
def fallback_tool(input: str) -> str:
    """Fallback LLM response when all else fails."""
    fallback_llm = ChatOpenAI(model="gpt-4.1-nano")  # Use cheaper or different model
    return fallback_llm.invoke(f"Fallback response for: {input}")

try:
    response = agent.run("I have a sore throat and fever")
except Exception as e:
    print("Primary agent failed, using fallback...")
    response = fallback_tool.run("I have a sore throat and fever")


<ipython-input-14-266132720>:11: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = agent.run("I have a sore throat and fever")




> Entering new AgentExecutor chain...

Invoking: `general` with `{'input': 'I have a sore throat and fever'}`


I'm sorry you're feeling unwell. A sore throat and fever can be caused by many things, including infections. If your fever is high, persistent, or if you have difficulty breathing, severe pain, or other concerning symptoms, please seek medical attention promptly. It's a good idea to see a healthcare professional to get an accurate diagnosis and appropriate care.Based on your symptoms of sore throat and fever, it could be caused by an infection or other factors. If your fever is high or persistent, or if you experience difficulty breathing or severe pain, please seek medical attention promptly. For now, rest, stay hydrated, and consider over-the-counter remedies to ease your symptoms. If your condition worsens or you have other concerning symptoms, see a healthcare professional.

> Finished chain.


In [19]:
def run_medibot(user_query: str) -> str:
    try:
        result = agent.invoke({"input": user_query})
        return result["output"] if isinstance(result, dict) and "output" in result else str(result)
    except Exception as e:
        print("⚠ Primary agent failed, using fallback...")
        return fallback_tool.run(user_query)



In [37]:
query=input("Hi how can I help : ")
response=run_medibot(query)
print(response)

Hi how can I help : wht is arinac for


> Entering new AgentExecutor chain...

Invoking: `get_drug_info` with `{'drug': 'Arinac'}`


⚠ Failed to fetch info for 'Arinac'. Reason: `run` not supported when there is not exactly one output key. Got ['result', 'source_documents'].It seems I couldn't retrieve specific information about "Arinac." Could you please double-check the spelling or provide more details? Alternatively, I can offer general advice if you describe what it's used for.

> Finished chain.
It seems I couldn't retrieve specific information about "Arinac." Could you please double-check the spelling or provide more details? Alternatively, I can offer general advice if you describe what it's used for.


In [ ]:
!pip install gradio

In [27]:
import gradio as gr

with gr.Blocks(css="""
    .gradio-container {
        height: 100vh;
        width: 100vw;
        padding: 0;
        margin: 0;
    }
    .chatbot {
        flex: 1;
        overflow: auto;
    }
""") as demo:
    gr.Markdown("<h1 style='text-align:center;'>🩺 MediBot</h1>")
    gr.Markdown("<p style='text-align:center;'>Ask me anything!</p>")

    chatbot = gr.Chatbot(elem_id="chatbot", height=500)

    with gr.Row():
        user_input = gr.Textbox(placeholder="Type your medical query here...", lines=1, scale=4)
        send_btn = gr.Button("Send", scale=1)

    def respond(message, history):
        response = run_medibot(message)
        history.append((message, response))
        return "", history

    send_btn.click(fn=respond, inputs=[user_input, chatbot], outputs=[user_input, chatbot])
    user_input.submit(fn=respond, inputs=[user_input, chatbot], outputs=[user_input, chatbot])  # ← Enter support

demo.launch()


<ipython-input-27-3829481262>:18: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(elem_id="chatbot", height=500)


It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a30aadf934fe6d5b54.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
